In [2]:
import sys
!{sys.executable} -m pip install -U pandas numpy matplotlib seaborn scikit-learn pillow opencv-python efficientnet-pytorch grad-cam

# Imports
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from efficientnet_pytorch import EfficientNet
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report, roc_auc_score, roc_curve
from sklearn.model_selection import train_test_split
from PIL import Image
from pathlib import Path
import cv2
import warnings
warnings.filterwarnings('ignore')

# PyTorch Grad-CAM
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image

print("✅ All imports successful!")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")


   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.7 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.7 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.7 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.7 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.7 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.7 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.7 MB ? eta -:--:--
   -- ------------------------------------- 0.5/9.7 MB 197.4 kB/s eta 0:00:47
   --- ------------------------------------ 0.8/9.7 MB 313.6 kB/s eta 0:00:29
   --- ------------------------------------ 0.8/9.7 MB 313.6 kB/s eta 0:00:29
   ----

In [3]:
class DeepfakeImageDataset(Dataset):
    """Custom dataset for deepfake images"""
    
    def __init__(self, root_dir, transform=None):
        self.root_dir = Path(root_dir)
        self.transform = transform
        self.samples = []
        
        # Load real and fake images
        for label_name, label_id in [('real', 0), ('fake', 1)]:
            label_dir = self.root_dir / label_name
            if label_dir.exists():
                for img_path in label_dir.glob("*.jpg"):
                    self.samples.append((img_path, label_id))
                for img_path in label_dir.glob("*.png"):
                    self.samples.append((img_path, label_id))
        
        print(f"Found {len(self.samples)} images: {sum(l[1] for l in self.samples)} fake, {len(self.samples)-sum(l[1] for l in self.samples)} real")
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        try:
            image = Image.open(img_path).convert('RGB')
        except:
            # Fallback for corrupted images
            image = Image.new('RGB', (224, 224), color='gray')
        
        if self.transform:
            image = self.transform(image)
        
        return image, label, str(img_path)

# Data transforms
train_transform = transforms.Compose([
    transforms.Resize((380, 380)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((380, 380)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

print("✅ Dataset class ready")


✅ Dataset class ready


In [4]:
# Create datasets
print("📊 Loading datasets...")
train_dataset = DeepfakeImageDataset('data/images/train', train_transform)
val_dataset = DeepfakeImageDataset('data/images/val', test_transform)
test_dataset = DeepfakeImageDataset('data/images/test', test_transform)

# DataLoaders
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)

print(f"✅ Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")
print(f"   Batches - Train: {len(train_loader)} | Val: {len(val_loader)} | Test: {len(test_loader)}")


📊 Loading datasets...
Found 4071 images: 4071 fake, 0 real
Found 586 images: 586 fake, 0 real
Found 5681 images: 5681 fake, 0 real
✅ Train: 4071 | Val: 586 | Test: 5681
   Batches - Train: 255 | Val: 19 | Test: 178


In [5]:
class EfficientNetDeepfake(nn.Module):
    def __init__(self, num_classes=2):
        super(EfficientNetDeepfake, self).__init__()
        # Load pretrained EfficientNet-B4
        self.base_model = EfficientNet.from_pretrained('efficientnet-b4')
        
        # Freeze early layers
        for param in list(self.base_model.parameters())[:-20]:
            param.requires_grad = False
        
        # Replace classifier
        in_features = self.base_model._fc.in_features
        self.base_model._fc = nn.Linear(in_features, num_classes)
        
        # Dropout for regularization
        self.dropout = nn.Dropout(0.5)
    
    def forward(self, x):
        x = self.base_model(x)
        return x

# Initialize model
model = EfficientNetDeepfake(num_classes=2).to(device)
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"✅ Model loaded: {total_params/1e6:.1f}M total params, {trainable_params/1e6:.1f}M trainable")
print(model.base_model._fc)


Loaded pretrained weights for efficientnet-b4
✅ Model loaded: 17.6M total params, 3.9M trainable
Linear(in_features=1792, out_features=2, bias=True)


In [ ]:
# Loss & Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=3, factor=0.5)

# Training function
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for batch_idx, (images, labels, _) in enumerate(loader):
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        
        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        
        if batch_idx % 10 == 0:
            print(f'\rBatch {batch_idx}/{len(loader)}, Loss: {loss.item():.4f}, Acc: {100.*correct/total:.1f}%', end='')
    
    return running_loss / len(loader), 100.*correct/total

# Validation function
def validate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    all_probs = []
    all_labels = []
    
    with torch.no_grad():
        for images, labels, _ in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            
            probs = torch.softmax(outputs, dim=1)[:,1]
            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    acc = 100.*correct/total
    auc_score = roc_auc_score(all_labels, all_probs)
    return running_loss / len(loader), acc, auc_score, all_probs, all_labels

print("\n🚀 Starting Training...\n")
best_auc = 0
train_losses, val_losses = [], []
train_accs, val_accs = [], []

for epoch in range(10):
    print(f"Epoch {epoch+1}/10")
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc, val_auc, _, _ = validate(model, val_loader, criterion, device)
    
    scheduler.step(val_loss)
    
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)
    
    print(f"  Train Loss: {train_loss:.4f}, Acc: {train_acc:.1f}%")
    print(f"  Val   Loss: {val_loss:.4f}, Acc: {val_acc:.1f}%, AUC: {val_auc:.3f}")
    
    # Save best model
    if val_auc > best_auc:
        best_auc = val_auc
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_auc': val_auc,
        }, 'models/image_efficientnet_b4_best.pth')
        print(f"  💾 New best model saved (AUC: {val_auc:.3f})")

print(f"\n✅ Training complete! Best Val AUC: {best_auc:.3f}")



🚀 Starting Training...

Epoch 1/10


In [ ]:
# Load best model
checkpoint = torch.load('models/image_efficientnet_b4_best.pth')
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
print(f"Loaded best model from epoch {checkpoint['epoch']}, Val AUC: {checkpoint['val_auc']:.3f}")

# Full test evaluation
test_loss, test_acc, test_auc, test_probs, test_labels = validate(model, test_loader, criterion, device)

print("\n📊 FINAL TEST RESULTS:")
print(f"Test Accuracy: {test_acc:.2f}%")
print(f"Test AUC-ROC:  {test_auc:.3f}")
print(f"Test Loss:     {test_loss:.4f}")

# Predictions for F1
test_preds = (np.array(test_probs) > 0.5).astype(int)
print(f"Test F1-Score: {f1_score(test_labels, test_preds):.3f}")

# Confusion Matrix
cm = confusion_matrix(test_labels, test_preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Real', 'Fake'], yticklabels=['Real', 'Fake'])
plt.title('Confusion Matrix (Test Set)')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.savefig('outputs/metrics/image_confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# ROC Curve
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
fpr, tpr, _ = roc_curve(test_labels, test_probs)
plt.plot(fpr, tpr, linewidth=3, label=f'EfficientNet-B4 (AUC={test_auc:.3f})')
plt.plot([0, 1], [0, 1], 'k--', alpha=0.5)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - Test Set')
plt.legend()
plt.grid(True, alpha=0.3)

# Precision-Recall
from sklearn.metrics import precision_recall_curve
plt.subplot(1, 2, 2)
precision, recall, _ = precision_recall_curve(test_labels, test_probs)
plt.plot(recall, precision, linewidth=3)
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('outputs/metrics/image_roc_pr_curves.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# Sample predictions with Grad-CAM
def visualize_gradcam(model, image_path, target_class=1):
    """Generate Grad-CAM for single image"""
    model.eval()
    
    # Load and preprocess image
    img = Image.open(image_path).convert('RGB')
    rgb_img = np.array(img) / 255.0
    input_tensor = test_transform(img).unsqueeze(0).to(device)
    
    # Grad-CAM
    grayscale_cam = cam(
        input_tensor=input_tensor,
        targets=[ClassifierOutputTarget(target_class)]
    )[0, :]
    
    # Overlay
    visualization = show_cam_on_image(rgb_img, grayscale_cam, use_rgb=True)
    
    # Prediction
    with torch.no_grad():
        output = model(input_tensor)
        prob = torch.softmax(output, dim=1)[0, target_class].item()
    
    return img, Image.fromarray(visualization), prob

# Initialize GradCAM
cam = GradCAM(model=model, target_layers=[model.base_model._conv_head])

# Test on sample fake images
fake_samples = list(Path('data/images/test/fake').glob('*.jpg'))[:4]
fig, axes = plt.subplots(2, 4, figsize=(20, 10))

for i, img_path in enumerate(fake_samples):
    orig_img, cam_img, prob = visualize_gradcam(model, img_path)
    
    axes[0, i].imshow(orig_img)
    axes[0, i].set_title(f'Original\nFake Prob: {prob:.3f}')
    axes[0, i].axis('off')
    
    axes[1, i].imshow(cam_img)
    axes[1, i].set_title('Grad-CAM Heatmap')
    axes[1, i].axis('off')

plt.suptitle('🖼️ EfficientNet-B4 + GradCAM: Fake Image Analysis', fontsize=16)
plt.tight_layout()
plt.savefig('outputs/heatmaps/gradcam_demo.png', dpi=300, bbox_inches='tight')
plt.show()
print("✅ Grad-CAM visualizations complete!")


In [ ]:
# Save final model
torch.save({
    'model_state_dict': model.state_dict(),
    'test_auc': test_auc,
    'test_accuracy': test_acc,
    'test_f1': f1_score(test_labels, test_preds),
}, 'models/image_efficientnet_b4_final.pth')

# Training curves
plt.figure(figsize=(15, 5))

plt.subplot(1, 2, 1)
plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Val Loss')
plt.title('Training & Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(train_accs, label='Train Acc')
plt.plot(val_accs, label='Val Acc')
plt.title('Training & Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy (%)')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('outputs/metrics/training_curves.png', dpi=300, bbox_inches='tight')
plt.show()

# Final results table
results_df = pd.DataFrame({
    'Metric': ['Accuracy', 'F1-Score', 'AUC-ROC', 'Best Val AUC'],
    'Value': [f"{test_acc:.3f}", f"{f1_score(test_labels, test_preds):.3f}", 
              f"{test_auc:.3f}", f"{best_auc:.3f}"]
})
print("\n🎉 FINAL RESULTS:")
print(results_df)
results_df.to_csv('outputs/metrics/image_final_results.csv', index=False)

print("\n✅ IMAGE MODULE COMPLETE!")
print("📁 Check: models/image_efficientnet_b4_final.pth")
print("📁 outputs/metrics/, outputs/heatmaps/")
print("Expected: 95-97% Test Accuracy! 🚀")
